In [4]:
import pandas as pd
import numpy as np
import xgboost as xgb
from datetime import datetime

def motor_proyeccion_gig(ruta_excel, marcas_a_predecir, fecha_proyeccion):
    print(f"--- 🚀 Iniciando Motor de Proyección para: {marcas_a_predecir} ---")
    
    # 1. CARGA Y LIMPIEZA
    df_info = pd.read_excel(ruta_excel, sheet_name='Info_Pdv')
    df_ventas = pd.read_excel(ruta_excel, sheet_name='Data_Historica')
    
    df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'], format='%d/%m/%Y', errors='coerce')
    df_master = pd.merge(df_ventas, df_info, on='punto_de_venta_id', how='left')
    
    # Limpieza de IDs y Creación de Ticket
    columnas_id = ['punto_de_venta_id', 'ubicacion_id', 'tipo_pdv_id', 'marca_id', 'categoria_id']
    for col in columnas_id:
        df_master[col] = pd.to_numeric(df_master[col].astype(str).str.strip().str[1:], errors='coerce').fillna(0).astype(int)
    
    df_master['ticket'] = np.where(df_master['transacciones'] > 0, df_master['ventas_monto'] / df_master['transacciones'], 0)
    
    # Filtros de Calidad
    df_master = df_master[(df_master['transacciones'] >= 1) & (df_master['ticket'] >= 1000) & (df_master['ticket'] <= 200000)].copy()

    # 2. INGENIERÍA TEMPORAL
    df_master = df_master.sort_values(by=['punto_de_venta_id', 'fecha'])
    fechas_pdv = df_master.groupby('punto_de_venta_id')['fecha'].agg(['min', 'max']).reset_index()
    
    grilla = []
    for _, row in fechas_pdv.iterrows():
        # Creamos el rango hasta la fecha de proyección
        rango = pd.date_range(start=row['min'], end=fecha_proyeccion, freq='MS')
        grilla.append(pd.DataFrame({'punto_de_venta_id': row['punto_de_venta_id'], 'fecha': rango}))
    
    df_model = pd.merge(pd.concat(grilla), df_master, on=['punto_de_venta_id', 'fecha'], how='left')
    
    # Rellenar info estática para las filas nuevas (Marzo)
    cols_estaticas = ['marca_id', 'categoria_id', 'tipo_pdv_id', 'ubicacion_id']
    df_model[cols_estaticas] = df_model.groupby('punto_de_venta_id')[cols_estaticas].ffill()
    
    # Lags y Features
    df_model['Mes'] = df_model['fecha'].dt.month
    df_model['ventas_lag_1'] = df_model.groupby('punto_de_venta_id')['ventas_monto'].shift(1).fillna(0)
    df_model['transacciones_lag_1'] = df_model.groupby('punto_de_venta_id')['transacciones'].shift(1).fillna(0)
    df_model['ticket_lag_1'] = df_model.groupby('punto_de_venta_id')['ticket'].shift(1).fillna(0)
    df_model['ventas_lag_12'] = df_model.groupby('punto_de_venta_id')['ventas_monto'].shift(12).fillna(0)
    df_model['meses_desde_apertura'] = df_model.groupby('punto_de_venta_id').cumcount() + 1
    df_model['ventas_roll_mean_3'] = df_model.groupby('punto_de_venta_id')['ventas_monto'].transform(
        lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()).fillna(0)

    # 3. ENTRENAMIENTO Y PREDICCIÓN
    # FILTRO CRÍTICO: El entrenamiento solo usa datos que TIENEN valor real (no NaNs)
    df_train = df_model[df_model['fecha'] < fecha_proyeccion].dropna(subset=['ventas_monto', 'transacciones', 'ticket'])
    
    # La data a predecir (Marzo)
    df_target = df_model[(df_model['fecha'] == fecha_proyeccion) & (df_model['marca_id'].isin(marcas_a_predecir))].copy()
    
    features = ['marca_id', 'categoria_id', 'tipo_pdv_id', 'ubicacion_id', 'Mes',
                'ventas_lag_1', 'transacciones_lag_1', 'ticket_lag_1', 
                'ventas_lag_12', 'meses_desde_apertura', 'ventas_roll_mean_3']
    
    targets = {'Ventas': 'ventas_monto', 'TX': 'transacciones', 'Ticket': 'ticket'}
    
    for nombre, col in targets.items():
        model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)
        # Entrenamos solo con la data limpia del pasado
        model.fit(df_train[features], df_train[col])
        # Predecimos el futuro
        df_target[f'Pred_{nombre}'] = np.maximum(0, model.predict(df_target[features]))

    # 4. EXPORTACIÓN
    cols_finales = ['fecha', 'punto_de_venta_id', 'marca_id', 'ubicacion_id', 'tipo_pdv_id', 'Pred_Ventas', 'Pred_TX', 'Pred_Ticket']
    df_export = df_target[cols_finales].copy()
    
    nombre_archivo = f"Proyeccion_{'_'.join(map(str, marcas_a_predecir))}_{fecha_proyeccion.strftime('%Y_%m')}.xlsx"
    df_export.to_excel(nombre_archivo, index=False)
    
    print(f"✅ Proyección completada exitosamente. Archivo: {nombre_archivo}")
    return df_export


In [5]:
# --- EJECUCIÓN DEL USUARIO ---
ruta = r"D:\Documents\Roger\Universidad\Proyecto de grado ll\DATA GIG\Base de datos - Proyecto Roger_encriptada febrero.xlsx"
marcas = [23, 27, 73]
fecha = pd.to_datetime('2026-03-01') # Ejemplo: Predecir Marzo

df_resultado = motor_proyeccion_gig(ruta, marcas, fecha)
display(df_resultado.head())

--- 🚀 Iniciando Motor de Proyección para: [23, 27, 73] ---
✅ Proyección completada exitosamente. Archivo: Proyeccion_23_27_73_2026_03.xlsx


,fecha,punto_de_venta_id,marca_id,ubicacion_id,tipo_pdv_id,Pred_Ventas,Pred_TX,Pred_Ticket
37883,2026-03-01,865,23.0,821.0,2.0,116133048.0,2002.678955,59748.042969
37982,2026-03-01,866,23.0,853.0,2.0,39413980.0,2676.231934,35730.863281
38167,2026-03-01,867,23.0,126.0,1.0,205892208.0,3653.698242,64594.023438
38278,2026-03-01,868,23.0,126.0,2.0,39947020.0,1862.097046,40840.523438
38462,2026-03-01,869,23.0,126.0,2.0,209518240.0,3800.157227,59082.414062


In [ ]:
import base64

with open("tu_logo_unad.png", "rb") as img_file:
    b64_string = base64.b64encode(img_file.read()).decode()
print(b64_string) # Copia todo ese texto largo que sale